<a href="https://www.kaggle.com/code/znine09/let-s-predict-airline-satisfaction-models-ipynb?scriptVersionId=354962428" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/playground-series-s6e10/sample_submission.csv
/kaggle/input/competitions/playground-series-s6e10/train.csv
/kaggle/input/competitions/playground-series-s6e10/test.csv


- **Uncomment the cells for running the relevant codes.**

In [2]:
TRAIN = pd.read_csv("/kaggle/input/competitions/playground-series-s6e10/train.csv")
TEST = pd.read_csv("/kaggle/input/competitions/playground-series-s6e10/test.csv")
SUBMIT = pd.read_csv("/kaggle/input/competitions/playground-series-s6e10/test.csv")

In [3]:
X, y = TRAIN.iloc[:, :-1], TRAIN.iloc[:, -1]

In [4]:
target = "satisfaction"
num_col = ["Age", "Flight Distance", "Departure Delay in Minutes", "Arrival Delay in Minutes"]
cat_col = [col for col in TRAIN.columns if col not in num_col and col != target and col != "id"]

nom_cat_col = ["Gender", "Customer Type", "Type of Travel"]
ord_cat_col = [col for col in cat_col if col not in nom_cat_col]

## Data preprocessing

In [5]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder

In [6]:
# Preprocessing pipeline

def preprocess_data(df, target_col=target, drop_nan_col="Arrival Delay in Minutes"):
    df_clean = df.dropna(subset=[drop_nan_col]).copy()
    
    nom_cat_col = ["Gender", "Customer Type", "Type of Travel"]
    nom_map = [df_clean[col].dropna().unique().tolist() for col in nom_cat_col]
    class_map = [['Eco', 'Eco Plus', 'Business']]
    
    preprocessor = ColumnTransformer(
        transformers=[
            ('nom', OrdinalEncoder(categories=nom_map), nom_cat_col),
            ('class', OrdinalEncoder(categories=class_map), ['Class'])
        ],
        remainder='passthrough',
        verbose_feature_names_out=False
    )
    
    X = df_clean.drop(columns=[target_col, "id"], errors='ignore')
    Y = df_clean[target_col].astype(int)
    
    # Transform and rebuild DataFrame restoring original column order
    X_processed = preprocessor.fit_transform(X)
    cols = preprocessor.get_feature_names_out()
    
    X_df = pd.DataFrame(X_processed, columns=cols, index=df_clean.index)[X.columns]
    
    return X_df, Y

In [7]:
# Usage
XP, YP = preprocess_data(TRAIN)

## Train / Test split

In [8]:
from sklearn.model_selection import train_test_split
X_train, X_valid, y_train, y_valid = train_test_split(XP, YP, stratify=YP, test_size=0.3, random_state=42)

## Trying different tree based models

In [9]:
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, precision_recall_fscore_support,
    roc_auc_score, classification_report, ConfusionMatrixDisplay
)

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

from imblearn.over_sampling import RandomOverSampler, SMOTE, SMOTENC


In [10]:
def evaluate_model(model, X_train, y_train, X_test, y_test, model_name="Classifier"):

    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    if hasattr(model, "predict_proba"):
        y_pred_prob = model.predict_proba(X_test)[:, 1]
    elif hasattr(model, "decision_function"):
        y_pred_prob = model.decision_function(X_test)
    else:
        y_pred_prob = None

    acc = accuracy_score(y_test, y_pred)
    bal_acc = balanced_accuracy_score(y_test, y_pred)
    prec, recall, f1, _ = precision_recall_fscore_support(
        y_test, y_pred, average="binary", zero_division=0
    )
    auc = (
        roc_auc_score(y_test, y_pred_prob) if y_pred_prob is not None else None
    )

    print(classification_report(y_test, y_pred, zero_division=0))
    print("-" * 40)

    fig, ax = plt.subplots(figsize=(4.5, 3))
    ConfusionMatrixDisplay.from_predictions(
        y_test, y_pred, display_labels=["No", "Yes"], ax=ax
    )
    ax.grid(False)
    plt.title(f"Confusion Matrix - {model_name}")
    plt.show()

    print("ROC-AUC score:", auc)

    metrics = {
        "Model/Condition": model_name,
        "Acc.": round(acc, 4),
        "Bal. Acc.": round(bal_acc, 4),
        "Prec.": round(prec, 4),
        "Recall": round(recall, 4),
        "F1": round(f1, 4),
        "ROC-AUC": round(auc, 4) if auc is not None else "N/A",
    }

    return model, metrics, {"y_test": y_test, "y_pred_prob": y_pred_prob}

### Decision Tree Classifer, Pruned Decision Tree, Random Forest, ExtraTrees, XGBoost, LightGBM, CatBoost (Baseline Testing, default parameters)

In [11]:
base_models = {
        "Baseline Tree": DecisionTreeClassifier(random_state=42),
        "Pruned Tree": DecisionTreeClassifier(max_depth=10, min_samples_leaf=5, random_state=42),
        "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
        "Extra Trees": ExtraTreesClassifier(n_estimators=100, random_state=42, n_jobs=-1),
        "XGBoost": XGBClassifier(n_estimators=100, learning_rate=0.1, random_state=42, eval_metric="logloss"),
        "LightGBM": LGBMClassifier(n_estimators=100, learning_rate=0.1, random_state=42, n_jobs=-1, verbosity=-1),
        "CatBoost": CatBoostClassifier(iterations=300, learning_rate=0.1, random_seed=42, verbose=0)
}

In [12]:
def evaluate_models(models, X_train, y_train, X_valid, y_valid):
    fit_results = []
    
    for name, model in models.items():
        model.fit(X_train, y_train)
        
        # Predict classes
        y_train_pred = model.predict(X_train)
        y_valid_pred = model.predict(X_valid)
        
        # Predict probabilities for ROC-AUC
        y_train_prob = model.predict_proba(X_train)[:, 1]
        y_valid_prob = model.predict_proba(X_valid)[:, 1]
        
        # Calculate scores
        train_acc = accuracy_score(y_train, y_train_pred)
        valid_acc = accuracy_score(y_valid, y_valid_pred)
        train_auc = roc_auc_score(y_train, y_train_prob)
        valid_auc = roc_auc_score(y_valid, y_valid_prob)
        
        fit_results.append({
            "Model": name,
            "Train Acc": round(train_acc, 4),
            "Valid Acc": round(valid_acc, 4),
            "Acc Gap": round(train_acc - valid_acc, 4),
            "Train ROC-AUC": round(train_auc, 4),
            "Valid ROC-AUC": round(valid_auc, 4),
            "AUC Gap": round(train_auc - valid_auc, 4)
        })
    
    fit_df = pd.DataFrame(fit_results)
    display(fit_df)
    
    # 3. Plot Train vs Validation Comparison
    x = np.arange(len(fit_df["Model"]))
    width = 0.35
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Accuracy Plot
    axes[0].bar(x - width/2, fit_df["Train Acc"], width, label="Train Acc", color="#1f77b4")
    axes[0].bar(x + width/2, fit_df["Valid Acc"], width, label="Valid Acc", color="#aec7e8")
    axes[0].set_title("Train vs Validation Accuracy")
    axes[0].set_xticks(x)
    axes[0].set_xticklabels(fit_df["Model"], rotation=25, ha="right")
    axes[0].set_ylim(0.8, 1.02)
    axes[0].legend()
    axes[0].grid(axis="y", linestyle="--", alpha=0.7)
    
    # ROC-AUC Plot
    axes[1].bar(x - width/2, fit_df["Train ROC-AUC"], width, label="Train ROC-AUC", color="#2ca02c")
    axes[1].bar(x + width/2, fit_df["Valid ROC-AUC"], width, label="Valid ROC-AUC", color="#98df8a")
    axes[1].set_title("Train vs Validation ROC-AUC")
    axes[1].set_xticks(x)
    axes[1].set_xticklabels(fit_df["Model"], rotation=25, ha="right")
    axes[1].set_ylim(0.8, 1.02)
    axes[1].legend()
    axes[1].grid(axis="y", linestyle="--", alpha=0.7)
    
    plt.tight_layout()
    plt.show()

In [13]:
# Using normal dataset without oversampling
# evaluate_models(base_models, X_train, y_train, X_valid, y_valid)

In [14]:
# Random Oversampling

# ros = RandomOverSampler(random_state=42)
# X_train_res, y_train_res = ros.fit_resample(X_train, y_train)

# evaluate_models(base_models, X_train_res, y_train_res, X_valid, y_valid)

In [15]:
# SMOTE
# smote = SMOTE(random_state=42)
# X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

# evaluate_models(base_models, X_train_sm, y_train_sm, X_valid, y_valid)

## Train using modified dataset, combining the variables arrival and departure delay

- **Since the correlation of the variables "Arrival Delay" and "Departure Delay" were almost high, these features are high, so we can combine those two features based. What I decided was the rate of delay = Arrival Delay - Departure Delay because:**

- **If (-ve): Pilot made up time in air**

- **If (+ve): Pilot lost time mid air**

- **If 0: No change in delay**,

- **This gives the directional context to deciidng split in tree.**

In [16]:
XP1 = XP.copy()

XP1["Inflight Delay Change"] = (XP1["Arrival Delay in Minutes"] - XP1["Departure Delay in Minutes"])
XP1 = XP1.drop(columns=["Arrival Delay in Minutes", "Departure Delay in Minutes"])

In [17]:
# print(XP1.columns)
# print(XP1.shape)
X_train1, X_valid1, y_train1, y_valid1 = train_test_split(XP1, YP, stratify=YP, test_size=0.3, random_state=42)

In [18]:
#Using modified dataset without oversampling
# evaluate_models(base_models, X_train1, y_train1, X_valid1, y_valid1)

In [19]:
# Random Oversampling
# ros1 = RandomOverSampler(random_state=42)
# X_train_res1, y_train_res1 = ros1.fit_resample(X_train1, y_train1)

# evaluate_models(base_models, X_train_res1, y_train_res1, X_valid1, y_valid1)

- **Finally considering all possible tree models, baseline performance is excelled for models LightGBM, XGBoost and CatBoost.**

- **And since the transformed dataset gives almost the same identical performance as the original dataset, we finalize to use this dataset.**

## 5 - Fold Stratified Cross validation

In [20]:
XP1.columns

Index(['Age', 'Flight Distance', 'Inflight wifi service',
       'Departure/Arrival time convenient', 'Ease of Online booking',
       'Gate location', 'Food and drink', 'Online boarding', 'Seat comfort',
       'Inflight entertainment', 'On-board service', 'Leg room service',
       'Baggage handling', 'Checkin service', 'Cleanliness', 'Gender',
       'Customer Type', 'Type of Travel', 'Class', 'Inflight Delay Change'],
      dtype='object')

In [21]:
num1_col = ["Age", "Flight Distance", 'Inflight Delay Change']
cat1_col = [col for col in XP1.columns if col not in num1_col]

for col in cat1_col:
    XP1[col] = XP1[col].astype('int64').astype('category')

In [22]:
XP1.info()

<class 'pandas.core.frame.DataFrame'>
Index: 699343 entries, 0 to 699634
Data columns (total 20 columns):
 #   Column                             Non-Null Count   Dtype   
---  ------                             --------------   -----   
 0   Age                                699343 non-null  float64 
 1   Flight Distance                    699343 non-null  float64 
 2   Inflight wifi service              699343 non-null  category
 3   Departure/Arrival time convenient  699343 non-null  category
 4   Ease of Online booking             699343 non-null  category
 5   Gate location                      699343 non-null  category
 6   Food and drink                     699343 non-null  category
 7   Online boarding                    699343 non-null  category
 8   Seat comfort                       699343 non-null  category
 9   Inflight entertainment             699343 non-null  category
 10  On-board service                   699343 non-null  category
 11  Leg room service               

In [23]:
from sklearn.model_selection import StratifiedKFold, cross_validate

In [24]:
models_1 = {
    "XGBoost": XGBClassifier(
        n_estimators=100,
        learning_rate=0.1,
        random_state=42,
        eval_metric="logloss",
        enable_categorical=True,
        tree_method="hist",
        n_jobs=-1,
    ),
    "LightGBM": LGBMClassifier(
        n_estimators=100,
        learning_rate=0.1,
        random_state=42,
        n_jobs=-1,
        verbosity=-1,
    ),
    "CatBoost": CatBoostClassifier(
        iterations=300,
        learning_rate=0.1,
        random_seed=42,
        verbose=0,
        thread_count=-1,
    ),
}

In [25]:
# skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# # Metrics to evaluate
# scoring = ["roc_auc", "accuracy", "precision", "recall", "f1"]

# fold_records = []
# summary_records = []

# for name, model in models_1.items():
#     print(f"Running 5-Fold Cross-Validation for {name}...")
#     fit_params = {"cat_features": cat1_col} if name == "CatBoost" else {}

#     # Run cross-validation
#     cv_results = cross_validate(model, XP1, YP, cv=skf, scoring=scoring, return_train_score=True, n_jobs=-1, params=fit_params)

#     # 1. Store fold-by-fold results
#     for fold_idx in range(skf.n_splits):
#         fold_records.append(
#             {
#                 "Model": name,
#                 "Fold": fold_idx + 1,
#                 "Train ROC-AUC": cv_results["train_roc_auc"][fold_idx],
#                 "Valid ROC-AUC": cv_results["test_roc_auc"][fold_idx],
#                 "Train Acc": cv_results["train_accuracy"][fold_idx],
#                 "Valid Acc": cv_results["test_accuracy"][fold_idx],
#                 "Valid F1": cv_results["test_f1"][fold_idx],
#             }
#         )

#     # 2. Store aggregated mean and std summary
#     summary_records.append(
#         {
#             "Model": name,
#             "Mean Valid ROC-AUC": np.mean(cv_results["test_roc_auc"]),
#             "Std Valid ROC-AUC": np.std(cv_results["test_roc_auc"]),
#             "Mean Valid Acc": np.mean(cv_results["test_accuracy"]),
#             "Std Valid Acc": np.std(cv_results["test_accuracy"]),
#             "Mean Valid F1": np.mean(cv_results["test_f1"]),
#             "Mean Fit Time (s)": np.mean(cv_results["fit_time"]),
#         }
#     )

# df_fold_results = pd.DataFrame(fold_records)
# df_summary = pd.DataFrame(summary_records).sort_values(
#     by="Mean Valid ROC-AUC", ascending=False
# )

In [26]:
# df_fold_results['ROC_AUC_Gap'] = (df_fold_results['Train ROC-AUC'] - df_fold_results['Valid ROC-AUC']).abs()

# top_3_least_overfit = df_fold_results.sort_values(by='ROC_AUC_Gap', ascending=True).head(3)

# display(top_3_least_overfit)

# # display(df_fold_results)
# display(df_summary)